In [ ]:
# from pyspark.sql import functions as F

# # Load data from table
# df = spark.table("public.stixor_iar_jul")

# # Define event date for analysis
# EVENT_DATE = "2024-07-30"

# # Filter and select base columns
# base = (
#     df.filter(F.col("data_date") <= F.lit(EVENT_DATE))
#     .select(
#         "customer_msisdn", "data_date", "trx_amt", "trx_status", "trx_channel", "trx_type",
#         "merchant_id", "reason_type", "pur_of_remit", "utility_company", "start_balance",
#         "end_balance", "trans_initiate_time", "ac_from", "ac_to", "bill_ref_number",
#         "fee", "fed", "trans_id", "ec"
#     )
#     .withColumn("data_date", F.to_date("data_date"))
# )

# # Create window flags and time-based features
# event_date = F.to_date(F.lit(EVENT_DATE))
# windowed = (
#     base
#     .withColumn("win_1d", F.col("data_date") >= F.date_sub(event_date, 1))
#     .withColumn("win_3d", F.col("data_date") >= F.date_sub(event_date, 3))
#     .withColumn("win_7d", F.col("data_date") >= F.date_sub(event_date, 7))
#     .withColumn("win_15d", F.col("data_date") >= F.date_sub(event_date, 15))
#     .withColumn("win_30d", F.col("data_date") >= F.date_sub(event_date, 30))
#     .withColumn("trx_hour", F.hour("trans_initiate_time"))
#     .withColumn(
#         "trx_time_bucket",
#         F.when((F.col("trx_hour") >= 0) & (F.col("trx_hour") < 6), "midnight")
#         .when((F.col("trx_hour") >= 6) & (F.col("trx_hour") < 12), "morning")
#         .when((F.col("trx_hour") >= 12) & (F.col("trx_hour") < 18), "afternoon")
#         .otherwise("evening")
#     )
# )

# # Optimize performance
# windowed = windowed.repartition("ac_from")
# windowed.cache()



# def generate_window_aggs(window_flag: str, hours: int):
#     """Generate aggregation functions for a specific time window."""
#     return [
#         # Basic transaction counts
#         F.count(F.when(F.col(window_flag), True)).alias(f"tx_count_{window_flag}"),
#         F.count(F.when(F.col(window_flag) & (F.col("trx_status") == "Completed"), True)).alias(f"tx_success_{window_flag}"),
#         F.count(F.when(F.col(window_flag) & (F.col("trx_status") != "Completed"), True)).alias(f"tx_failed_{window_flag}"),
#         F.countDistinct(F.when(F.col(window_flag), F.col("data_date"))).alias(f"active_days_{window_flag}"),
#         (F.max(F.when(F.col(window_flag), F.col("trans_initiate_time"))) -
#          F.min(F.when(F.col(window_flag), F.col("trans_initiate_time")))).alias(f"tx_span_{window_flag}"),

#         # Transaction amount statistics
#         F.sum(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"sum_trx_amt_{window_flag}"),
#         F.avg(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"avg_trx_amt_{window_flag}"),
#         F.max(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"max_trx_amt_{window_flag}"),
#         F.min(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"min_trx_amt_{window_flag}"),
#         F.stddev(F.when(F.col(window_flag), F.col("trx_amt"))).alias(f"stddev_trx_amt_{window_flag}"),

#         # Balance statistics
#         F.avg(F.when(F.col(window_flag), F.col("start_balance"))).alias(f"avg_start_balance_{window_flag}"),
#         F.avg(F.when(F.col(window_flag), F.col("end_balance"))).alias(f"avg_end_balance_{window_flag}"),
#         F.avg(F.when(F.col(window_flag), F.col("end_balance") - F.col("start_balance"))).alias(f"avg_balance_change_{window_flag}"),
#         F.sum(F.when(F.col(window_flag), F.col("end_balance") - F.col("start_balance"))).alias(f"total_balance_change_{window_flag}"),

#         # Unique counts for diversity metrics
#         F.countDistinct(F.when(F.col(window_flag), F.col("trx_channel"))).alias(f"unique_channels_{window_flag}"),
#         F.countDistinct(F.when(F.col(window_flag), F.col("trx_type"))).alias(f"unique_types_{window_flag}"),
#         F.countDistinct(F.when(F.col(window_flag), F.col("merchant_id"))).alias(f"unique_merchants_{window_flag}"),
#         F.countDistinct(F.when(F.col(window_flag), F.col("reason_type"))).alias(f"unique_reason_types_{window_flag}"),
#         F.countDistinct(F.when(F.col(window_flag), F.col("pur_of_remit"))).alias(f"unique_purposes_{window_flag}"),
#         F.countDistinct(F.when(F.col(window_flag), F.col("utility_company"))).alias(f"unique_utilities_{window_flag}"),

#         # Failure and risk ratios
#         (F.count(F.when(F.col(window_flag) & (F.col("trx_status") != "Completed"), True)).cast("float") /
#          F.when(F.count(F.when(F.col(window_flag), True)) != 0,
#                 F.count(F.when(F.col(window_flag), True)))).alias(f"failure_ratio_{window_flag}"),

#         # High-value transaction metrics
#         F.count(F.when(F.col(window_flag) & (F.col("trx_amt") > 100000), True)).alias(f"high_value_count_{window_flag}"),
#         (F.count(F.when(F.col(window_flag) & (F.col("trx_amt") > 100000), True)).cast("float") /
#          F.when(F.count(F.when(F.col(window_flag), True)) != 0,
#                 F.count(F.when(F.col(window_flag), True)))).alias(f"high_value_ratio_{window_flag}"),

#         # Average hourly metrics
#         (F.count(F.when(F.col(window_flag), True)) / F.lit(hours)).alias(f"avg_hourly_tx_count_{window_flag}"),
#         (F.sum(F.when(F.col(window_flag), F.col("trx_amt"))) / F.lit(hours)).alias(f"avg_hourly_tx_amt_{window_flag}")
#     ]


# # Generate aggregations for all time windows
# aggregations = []
# windows = [("win_1d", 24), ("win_3d", 72), ("win_7d", 168), ("win_15d", 360), ("win_30d", 720)]
# for win_flag, hrs in windows:
#     aggregations.extend(generate_window_aggs(win_flag, hrs))

# # Apply aggregations and write results
# agg = windowed.groupBy("ac_from").agg(*aggregations)
# agg.write.mode("overwrite").parquet("/mnt/fraud/features_agg/")


# -- Aggregated customer-level features for multiple time windows
# WITH base AS (
#     SELECT
#         customer_msisdn,
#         data_date,
#         trx_amt,
#         trx_status,
#         trx_channel,
#         trx_type,
#         merchant_id,
#         reason_type,
#         pur_of_remit,
#         utility_company,
#         start_balance,
#         end_balance,
#         trans_initiate_time,
#         ac_from,
#         ac_to,
#         bill_ref_number,
#         fee,
#         fed,
#         trans_id,
#         ec
#     FROM public.stixor_iar_mbar_20250701_sample
#     WHERE customer_msisdn IS NOT NULL
# ),
# windowed AS (
#     SELECT
#         customer_msisdn,
#         data_date,
#         trx_amt,
#         trx_status,
#         trx_channel,
#         trx_type,
#         merchant_id,
#         reason_type,
#         pur_of_remit,
#         utility_company,
#         start_balance,
#         end_balance,
#         trans_initiate_time,
#         ac_from,
#         ac_to,
#         bill_ref_number,
#         fee,
#         fed,
#         trans_id,
#         ec,
#         -- Window boundaries
#         data_date >= CURRENT_DATE - INTERVAL '1 day' AS win_1d,
#         data_date >= CURRENT_DATE - INTERVAL '3 day' AS win_3d,
#         data_date >= CURRENT_DATE - INTERVAL '7 day' AS win_7d,
#         data_date >= CURRENT_DATE - INTERVAL '10 day' AS win_10d,
#         data_date >= CURRENT_DATE - INTERVAL '15 day' AS win_15d,
#         data_date >= CURRENT_DATE - INTERVAL '30 day' AS win_30d
#     FROM base
# )
# SELECT
#     customer_msisdn,

#     -- 1d window
#     COUNT(*) FILTER (WHERE win_1d) AS tx_count_1d,
#     COUNT(*) FILTER (WHERE win_1d AND trx_status = 'SUCCESS') AS tx_success_1d,
#     COUNT(*) FILTER (WHERE win_1d AND trx_status != 'SUCCESS') AS tx_failed_1d,
#     COUNT(DISTINCT data_date) FILTER (WHERE win_1d) AS active_days_1d,
#     (MAX(trans_initiate_time) FILTER (WHERE win_1d) - MIN(trans_initiate_time) FILTER (WHERE win_1d)) AS tx_span_1d,
#     SUM(trx_amt) FILTER (WHERE win_1d) AS sum_trx_amt_1d,
#     AVG(trx_amt) FILTER (WHERE win_1d) AS avg_trx_amt_1d,
#     MAX(trx_amt) FILTER (WHERE win_1d) AS max_trx_amt_1d,
#     MIN(trx_amt) FILTER (WHERE win_1d) AS min_trx_amt_1d,
#     STDDEV(trx_amt) FILTER (WHERE win_1d) AS stddev_trx_amt_1d,
#     AVG(start_balance) FILTER (WHERE win_1d) AS avg_start_balance_1d,
#     AVG(end_balance) FILTER (WHERE win_1d) AS avg_end_balance_1d,
#     AVG(end_balance - start_balance) FILTER (WHERE win_1d) AS avg_balance_change_1d,
#     SUM(end_balance - start_balance) FILTER (WHERE win_1d) AS total_balance_change_1d,
#     COUNT(DISTINCT trx_channel) FILTER (WHERE win_1d) AS unique_channels_1d,
#     COUNT(DISTINCT trx_type) FILTER (WHERE win_1d) AS unique_types_1d,
#     COUNT(DISTINCT merchant_id) FILTER (WHERE win_1d) AS unique_merchants_1d,
#     COUNT(DISTINCT reason_type) FILTER (WHERE win_1d) AS unique_reason_types_1d,
#     COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_1d) AS unique_purposes_1d,
#     COUNT(DISTINCT utility_company) FILTER (WHERE win_1d) AS unique_utilities_1d,
#     COUNT(*) FILTER (WHERE win_1d AND trx_status != 'SUCCESS')::float / NULLIF(COUNT(*) FILTER (WHERE win_1d), 0) AS failure_ratio_1d,
#     COUNT(*) FILTER (WHERE win_1d AND trx_amt > 100000) AS high_value_count_1d,
#     COUNT(*) FILTER (WHERE win_1d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_1d), 0) AS high_value_ratio_1d,

#     -- Repeat for 3d, 7d, 10d, 15d, 30d windows
#     COUNT(*) FILTER (WHERE win_3d) AS tx_count_3d,
#     COUNT(*) FILTER (WHERE win_3d AND trx_status = 'SUCCESS') AS tx_success_3d,
#     COUNT(*) FILTER (WHERE win_3d AND trx_status != 'SUCCESS') AS tx_failed_3d,
#     COUNT(DISTINCT data_date) FILTER (WHERE win_3d) AS active_days_3d,
#     (MAX(trans_initiate_time) FILTER (WHERE win_3d) - MIN(trans_initiate_time) FILTER (WHERE win_3d)) AS tx_span_3d,
#     SUM(trx_amt) FILTER (WHERE win_3d) AS sum_trx_amt_3d,
#     AVG(trx_amt) FILTER (WHERE win_3d) AS avg_trx_amt_3d,
#     MAX(trx_amt) FILTER (WHERE win_3d) AS max_trx_amt_3d,
#     MIN(trx_amt) FILTER (WHERE win_3d) AS min_trx_amt_3d,
#     STDDEV(trx_amt) FILTER (WHERE win_3d) AS stddev_trx_amt_3d,
#     AVG(start_balance) FILTER (WHERE win_3d) AS avg_start_balance_3d,
#     AVG(end_balance) FILTER (WHERE win_3d) AS avg_end_balance_3d,
#     AVG(end_balance - start_balance) FILTER (WHERE win_3d) AS avg_balance_change_3d,
#     SUM(end_balance - start_balance) FILTER (WHERE win_3d) AS total_balance_change_3d,
#     COUNT(DISTINCT trx_channel) FILTER (WHERE win_3d) AS unique_channels_3d,
#     COUNT(DISTINCT trx_type) FILTER (WHERE win_3d) AS unique_types_3d,
#     COUNT(DISTINCT merchant_id) FILTER (WHERE win_3d) AS unique_merchants_3d,
#     COUNT(DISTINCT reason_type) FILTER (WHERE win_3d) AS unique_reason_types_3d,
#     COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_3d) AS unique_purposes_3d,
#     COUNT(DISTINCT utility_company) FILTER (WHERE win_3d) AS unique_utilities_3d,
#     COUNT(*) FILTER (WHERE win_3d AND trx_status != 'SUCCESS')::float / NULLIF(COUNT(*) FILTER (WHERE win_3d), 0) AS failure_ratio_3d,
#     COUNT(*) FILTER (WHERE win_3d AND trx_amt > 100000) AS high_value_count_3d,
#     COUNT(*) FILTER (WHERE win_3d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_3d), 0) AS high_value_ratio_3d,

#     -- Repeat for 7d, 10d, 15d, 30d windows (same pattern as above)
#     COUNT(*) FILTER (WHERE win_7d) AS tx_count_7d,
#     COUNT(*) FILTER (WHERE win_7d AND trx_status = 'SUCCESS') AS tx_success_7d,
#     COUNT(*) FILTER (WHERE win_7d AND trx_status != 'SUCCESS') AS tx_failed_7d,
#     COUNT(DISTINCT data_date) FILTER (WHERE win_7d) AS active_days_7d,
#     (MAX(trans_initiate_time) FILTER (WHERE win_7d) - MIN(trans_initiate_time) FILTER (WHERE win_7d)) AS tx_span_7d,
#     SUM(trx_amt) FILTER (WHERE win_7d) AS sum_trx_amt_7d,
#     AVG(trx_amt) FILTER (WHERE win_7d) AS avg_trx_amt_7d,
#     MAX(trx_amt) FILTER (WHERE win_7d) AS max_trx_amt_7d,
#     MIN(trx_amt) FILTER (WHERE win_7d) AS min_trx_amt_7d,
#     STDDEV(trx_amt) FILTER (WHERE win_7d) AS stddev_trx_amt_7d,
#     AVG(start_balance) FILTER (WHERE win_7d) AS avg_start_balance_7d,
#     AVG(end_balance) FILTER (WHERE win_7d) AS avg_end_balance_7d,
#     AVG(end_balance - start_balance) FILTER (WHERE win_7d) AS avg_balance_change_7d,
#     SUM(end_balance - start_balance) FILTER (WHERE win_7d) AS total_balance_change_7d,
#     COUNT(DISTINCT trx_channel) FILTER (WHERE win_7d) AS unique_channels_7d,
#     COUNT(DISTINCT trx_type) FILTER (WHERE win_7d) AS unique_types_7d,
#     COUNT(DISTINCT merchant_id) FILTER (WHERE win_7d) AS unique_merchants_7d,
#     COUNT(DISTINCT reason_type) FILTER (WHERE win_7d) AS unique_reason_types_7d,
#     COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_7d) AS unique_purposes_7d,
#     COUNT(DISTINCT utility_company) FILTER (WHERE win_7d) AS unique_utilities_7d,
#     COUNT(*) FILTER (WHERE win_7d AND trx_status != 'SUCCESS')::float / NULLIF(COUNT(*) FILTER (WHERE win_7d), 0) AS failure_ratio_7d,
#     COUNT(*) FILTER (WHERE win_7d AND trx_amt > 100000) AS high_value_count_7d,
#     COUNT(*) FILTER (WHERE win_7d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_7d), 0) AS high_value_ratio_7d,

#     COUNT(*) FILTER (WHERE win_10d) AS tx_count_10d,
#     COUNT(*) FILTER (WHERE win_10d AND trx_status = 'SUCCESS') AS tx_success_10d,
#     COUNT(*) FILTER (WHERE win_10d AND trx_status != 'SUCCESS') AS tx_failed_10d,
#     COUNT(DISTINCT data_date) FILTER (WHERE win_10d) AS active_days_10d,
#     (MAX(trans_initiate_time) FILTER (WHERE win_10d) - MIN(trans_initiate_time) FILTER (WHERE win_10d)) AS tx_span_10d,
#     SUM(trx_amt) FILTER (WHERE win_10d) AS sum_trx_amt_10d,
#     AVG(trx_amt) FILTER (WHERE win_10d) AS avg_trx_amt_10d,
#     MAX(trx_amt) FILTER (WHERE win_10d) AS max_trx_amt_10d,
#     MIN(trx_amt) FILTER (WHERE win_10d) AS min_trx_amt_10d,
#     STDDEV(trx_amt) FILTER (WHERE win_10d) AS stddev_trx_amt_10d,
#     AVG(start_balance) FILTER (WHERE win_10d) AS avg_start_balance_10d,
#     AVG(end_balance) FILTER (WHERE win_10d) AS avg_end_balance_10d,
#     AVG(end_balance - start_balance) FILTER (WHERE win_10d) AS avg_balance_change_10d,
#     SUM(end_balance - start_balance) FILTER (WHERE win_10d) AS total_balance_change_10d,
#     COUNT(DISTINCT trx_channel) FILTER (WHERE win_10d) AS unique_channels_10d,
#     COUNT(DISTINCT trx_type) FILTER (WHERE win_10d) AS unique_types_10d,
#     COUNT(DISTINCT merchant_id) FILTER (WHERE win_10d) AS unique_merchants_10d,
#     COUNT(DISTINCT reason_type) FILTER (WHERE win_10d) AS unique_reason_types_10d,
#     COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_10d) AS unique_purposes_10d,
#     COUNT(DISTINCT utility_company) FILTER (WHERE win_10d) AS unique_utilities_10d,
#     COUNT(*) FILTER (WHERE win_10d AND trx_status != 'SUCCESS')::float / NULLIF(COUNT(*) FILTER (WHERE win_10d), 0) AS failure_ratio_10d,
#     COUNT(*) FILTER (WHERE win_10d AND trx_amt > 100000) AS high_value_count_10d,
#     COUNT(*) FILTER (WHERE win_10d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_10d), 0) AS high_value_ratio_10d,

#     COUNT(*) FILTER (WHERE win_15d) AS tx_count_15d,
#     COUNT(*) FILTER (WHERE win_15d AND trx_status = 'SUCCESS') AS tx_success_15d,
#     COUNT(*) FILTER (WHERE win_15d AND trx_status != 'SUCCESS') AS tx_failed_15d,
#     COUNT(DISTINCT data_date) FILTER (WHERE win_15d) AS active_days_15d,
#     (MAX(trans_initiate_time) FILTER (WHERE win_15d) - MIN(trans_initiate_time) FILTER (WHERE win_15d)) AS tx_span_15d,
#     SUM(trx_amt) FILTER (WHERE win_15d) AS sum_trx_amt_15d,
#     AVG(trx_amt) FILTER (WHERE win_15d) AS avg_trx_amt_15d,
#     MAX(trx_amt) FILTER (WHERE win_15d) AS max_trx_amt_15d,
#     MIN(trx_amt) FILTER (WHERE win_15d) AS min_trx_amt_15d,
#     STDDEV(trx_amt) FILTER (WHERE win_15d) AS stddev_trx_amt_15d,
#     AVG(start_balance) FILTER (WHERE win_15d) AS avg_start_balance_15d,
#     AVG(end_balance) FILTER (WHERE win_15d) AS avg_end_balance_15d,
#     AVG(end_balance - start_balance) FILTER (WHERE win_15d) AS avg_balance_change_15d,
#     SUM(end_balance - start_balance) FILTER (WHERE win_15d) AS total_balance_change_15d,
#     COUNT(DISTINCT trx_channel) FILTER (WHERE win_15d) AS unique_channels_15d,
#     COUNT(DISTINCT trx_type) FILTER (WHERE win_15d) AS unique_types_15d,
#     COUNT(DISTINCT merchant_id) FILTER (WHERE win_15d) AS unique_merchants_15d,
#     COUNT(DISTINCT reason_type) FILTER (WHERE win_15d) AS unique_reason_types_15d,
#     COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_15d) AS unique_purposes_15d,
#     COUNT(DISTINCT utility_company) FILTER (WHERE win_15d) AS unique_utilities_15d,
#     COUNT(*) FILTER (WHERE win_15d AND trx_status != 'SUCCESS')::float / NULLIF(COUNT(*) FILTER (WHERE win_15d), 0) AS failure_ratio_15d,
#     COUNT(*) FILTER (WHERE win_15d AND trx_amt > 100000) AS high_value_count_15d,
#     COUNT(*) FILTER (WHERE win_15d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_15d), 0) AS high_value_ratio_15d,

#     COUNT(*) FILTER (WHERE win_30d) AS tx_count_30d,
#     COUNT(*) FILTER (WHERE win_30d AND trx_status = 'SUCCESS') AS tx_success_30d,
#     COUNT(*) FILTER (WHERE win_30d AND trx_status != 'SUCCESS') AS tx_failed_30d,
#     COUNT(DISTINCT data_date) FILTER (WHERE win_30d) AS active_days_30d,
#     (MAX(trans_initiate_time) FILTER (WHERE win_30d) - MIN(trans_initiate_time) FILTER (WHERE win_30d)) AS tx_span_30d,
#     SUM(trx_amt) FILTER (WHERE win_30d) AS sum_trx_amt_30d,
#     AVG(trx_amt) FILTER (WHERE win_30d) AS avg_trx_amt_30d,
#     MAX(trx_amt) FILTER (WHERE win_30d) AS max_trx_amt_30d,
#     MIN(trx_amt) FILTER (WHERE win_30d) AS min_trx_amt_30d,
#     STDDEV(trx_amt) FILTER (WHERE win_30d) AS stddev_trx_amt_30d,
#     AVG(start_balance) FILTER (WHERE win_30d) AS avg_start_balance_30d,
#     AVG(end_balance) FILTER (WHERE win_30d) AS avg_end_balance_30d,
#     AVG(end_balance - start_balance) FILTER (WHERE win_30d) AS avg_balance_change_30d,
#     SUM(end_balance - start_balance) FILTER (WHERE win_30d) AS total_balance_change_30d,
#     COUNT(DISTINCT trx_channel) FILTER (WHERE win_30d) AS unique_channels_30d,
#     COUNT(DISTINCT trx_type) FILTER (WHERE win_30d) AS unique_types_30d,
#     COUNT(DISTINCT merchant_id) FILTER (WHERE win_30d) AS unique_merchants_30d,
#     COUNT(DISTINCT reason_type) FILTER (WHERE win_30d) AS unique_reason_types_30d,
#     COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_30d) AS unique_purposes_30d,
#     COUNT(DISTINCT utility_company) FILTER (WHERE win_30d) AS unique_utilities_30d,
#     COUNT(*) FILTER (WHERE win_30d AND trx_status != 'SUCCESS')::float / NULLIF(COUNT(*) FILTER (WHERE win_30d), 0) AS failure_ratio_30d,
#     COUNT(*) FILTER (WHERE win_30d AND trx_amt > 100000) AS high_value_count_30d,
#     COUNT(*) FILTER (WHERE win_30d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_30d), 0) AS high_value_ratio_30d

# FROM windowed
# GROUP BY customer_msisdn

In [1]:
from pyspark.sql import SparkSession
from datetime import datetime, timedelta
import os

# Database configuration
DB_CONFIG = {
    'host': '10.205.161.118',
    'port': '5432',
    'database': 'db_fraud',
    'user': 'dfstechbi',
    'password': 'DfsTeChB1@923'
}

# JDBC Configuration
jdbc_driver_path = "/root/research-dir/dev/jazzcash-fraud-detection/scripts/postgresql-42.7.1.jar"
jdbc_url = f"jdbc:postgresql://{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

# Optimized JDBC properties
properties = {
    "user": DB_CONFIG['user'],
    "password": DB_CONFIG['password'],
    "driver": "org.postgresql.Driver",
    "fetchsize": "10000",
    "batchsize": "15000",
    "isolationLevel": "READ_UNCOMMITTED",
    "queryTimeout": "1200",
    "loginTimeout": "60",
    "socketTimeout": "1200",
    "tcpKeepAlive": "true",
    "prepareThreshold": "5",
    "reWriteBatchedInserts": "true",
    "defaultRowFetchSize": "10000"
}

table_name = "public.stixor_iar_jul"

print("🔧 Setting up date-based predicates...")

# Create predicates for optimized partitioning
predicates = []
start_date = datetime.strptime("2025-07-01", "%Y-%m-%d")
end_date = datetime.strptime("2025-07-31", "%Y-%m-%d")

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    predicates.append(f"data_date = '{date_str}'")
    current_date += timedelta(days=1)

print(f"📊 Created {len(predicates)} predicates for date range")

print("🚀 Creating optimized Spark session...")

# Create Spark session with comprehensive configuration
spark = SparkSession.builder \
    .appName("Load-IAR-July-Data-Optimized") \
    .master("spark://localhost:7077") \
    .config("spark.jars", jdbc_driver_path) \
    .config("spark.executor.memory", "20g") \
    .config("spark.executor.memoryOverhead", "1g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "80") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "10000") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.sql.broadcastTimeout", "600s") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print("✅ Spark session created successfully!")
print(f"📱 Application ID: {spark.sparkContext.applicationId}")
print(f"🎯 Master: {spark.sparkContext.master}")

🔧 Setting up date-based predicates...
📊 Created 31 predicates for date range
🚀 Creating optimized Spark session...


25/10/15 18:56:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/15 18:56:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Spark session created successfully!
📱 Application ID: app-20251015185603-0059
🎯 Master: spark://localhost:7077


In [ ]:
print("\n🔗 Loading data from PostgreSQL...")

# Load data using predicate-based partitioning
df = spark.read.jdbc(
    url=jdbc_url,
    table=table_name,
    predicates=predicates,
    properties=properties
)

print("✅ Data loading configured successfully!")
print(f"📈 DataFrame partitions: {df.rdd.getNumPartitions()}")

# Cache for better performance
df.cache()
print("💾 DataFrame cached for optimized access")

print("\n📋 Basic DataFrame Info:")
print(f"🔢 Total partitions: {df.rdd.getNumPartitions()}")
print("📊 Schema preview:")
for field in df.schema.fields[:5]:
    print(f"  - {field.name}: {field.dataType}")

print(f"\n🎯 Ready to process fraud data for July 2025!")

In [2]:
df.groupBy('data_date').count().show()

+----------+--------+
| data_date|   count|
+----------+--------+
|2025-07-01|12546035|
|2025-07-02|13204748|
|2025-07-03|13375098|
|2025-07-04|12570037|
|2025-07-05|11150252|
|2025-07-06| 8757251|
|2025-07-07|13467970|
|2025-07-08|13363032|
|2025-07-09|13372656|
|2025-07-10|13322670|
|2025-07-11|13379387|
|2025-07-12|13532458|
|2025-07-13|12765033|
|2025-07-14|13706248|
|2025-07-15|13677842|
|2025-07-16|13804209|
|2025-07-17|13576565|
|2025-07-18|12923723|
|2025-07-19|13366922|
|2025-07-20|12659197|
+----------+--------+
only showing top 20 rows


In [ ]:
import time 
start_time = time.time()
df_july_senders = df.select("ac_from").distinct()
df_july_senders.cache()
july_senders_count = df_july_senders.count()
print(f"{july_senders_count:,} unique sending customers in July")
print(f"Completed in {(time.time() - start_time)/60:.2f} minutes\n")

22,947,201 unique sending customers in July
Completed in 1429.51 seconds



In [5]:
print(f"Number of partitions in df_july_senders: {df_july_senders.rdd.getNumPartitions()}")

Number of partitions in df_july_senders: 200


In [6]:
df_july_senders = df_july_senders.coalesce(23)

# OPTION A: Write to Parquet (recommended for further processing)
output_path = "../data/sender_customer_july_2025"

df_july_senders.write \
    .mode("overwrite") \
    .parquet(output_path)

In [8]:
import time 
start_time = time.time()
df_july_receivers = df.select("ac_to").distinct()
df_july_receivers.cache()
july_receiver_count = df_july_receivers.count()
print(f"{july_receiver_count:,} unique receiving customers in July")
print(f"Completed in {(time.time() - start_time)/60:.2f} minutes\n")

19,452,571 unique receiving customers in July
Completed in 0.78 minutes



In [9]:
print(f"Number of partitions in df_july_receivers: {df_july_receivers.rdd.getNumPartitions()}")

Number of partitions in df_july_receivers: 200


In [10]:
df_july_receivers = df_july_receivers.coalesce(20)

output_path = "../data/receiver_customer_july_2025"

df_july_receivers.write \
    .mode("overwrite") \
    .parquet(output_path)

# Account Types for July Transaction Data

## July Senders (ac_from)

In [2]:
# OPTIMIZED: Analysis for df_july_senders with batch processing and parallelism
import math
from pyspark.sql.functions import col

stixor_mbar_v_table = "public.stixor_mbar_v"

print("🚀 Starting optimized batch processing for July senders...")

# Step 1: Get count and implement batch processing
# Load July senders from parquet file instead of counting DataFrame
df_july_senders = spark.read.parquet("../data/sender_customer_july_2025")
july_senders_count = df_july_senders.count()
print(f"📊 Total July senders loaded from parquet: {july_senders_count:,}")

🚀 Starting optimized batch processing for July senders...


📊 Total July senders loaded from parquet: 22,947,201


In [9]:
# Configure batch parameters
BATCH_SIZE = 30000  # Process 5K accounts per batch to avoid memory issues
total_batches = math.ceil(july_senders_count / BATCH_SIZE)
print(f"🔄 Will process in {total_batches} batches of {BATCH_SIZE} accounts each")

🔄 Will process in 765 batches of 30000 accounts each


In [10]:
# Use SQL pushdown to filter account_type_name in the JDBC query itself
sql_query_customer = f"""
(SELECT DISTINCT a_c_reference 
    FROM {stixor_mbar_v_table} 
    WHERE account_type_name = 'Customer Account'
) AS customer_accounts
"""

df_mbar_customer_a_c_reference = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_customer,
    properties=properties
).distinct()

output_path_mbar_customer_a_c_reference = "../data/mbar_customer_a_c_reference"
df_mbar_customer_a_c_reference.write.mode("overwrite").parquet(output_path_mbar_customer_a_c_reference)

print(f"✅ Saved distinct a_c_reference for Customer Account to {output_path_mbar_customer_a_c_reference}")

✅ Saved distinct a_c_reference for Customer Account to ../data/mbar_customer_a_c_reference


In [ ]:
print(f"Count of df_mbar_customer_a_c_reference: {df_mbar_customer_a_c_reference.count():,}")b

In [ ]:
df_mbar_customer_a_c_reference = spark.read.parquet(output_path_mbar_customer_a_c_reference)
print(f"Count from parquet: {df_mbar_customer_a_c_reference.count():,}")

In [6]:
# Take 10 accounts from df_july_senders
sample_accounts = [row.ac_from for row in df_july_senders.limit(10000).collect()]
sample_accounts_clean = [str(ac).replace("'", "''") for ac in sample_accounts]
sample_in_clause = "'" + "','".join(sample_accounts_clean) + "'"

sample_query = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({sample_in_clause})) as sample_mbar
"""

sample_properties = {
    **properties,
    "fetchsize": "100",
    "batchsize": "100",
    "queryTimeout": "60",
    "socketTimeout": "60",
    "defaultRowFetchSize": "100",
    "prepareThreshold": "1",
    "reWriteBatchedInserts": "true"
}

df_sample_accounts = spark.read.jdbc(
    url=jdbc_url,
    table=sample_query,
    properties=sample_properties
)

df_sample_accounts.show()

+--------------------+-----------------+
|       a_c_reference|account_type_name|
+--------------------+-----------------+
|gn5XLdLLOBQDOytuZ...| Customer Account|
|J0b1vr56VDCr3KynM...| Customer Account|
|i6U/F+1Owo9rOkU0i...| Customer Account|
|4CPpr822jIXcpu5sh...| Customer Account|
|wl6P1OrFyM+wDHskP...| Customer Account|
|QQKv5wUT2uneOK3ym...| Customer Account|
|wyB441kggIop+Tdup...| Customer Account|
|xDcDmBuqdIkt4CcKv...| Customer Account|
|Bu+lDkAbeNvJ5f+XY...| Customer Account|
|csI2jRfdbFIEOiocI...| Customer Account|
|AhrszPlgIFeDkyfNi...| Customer Account|
|JOOqzVoKQUUCA6AJw...| Customer Account|
|Ty4VFu3prP+Oix44q...| Customer Account|
|t4Hf/4hCc8a7J1gFP...| Customer Account|
|nyOemgTQmzN+GC0Lj...| Customer Account|
|53ZEmjo/5VYFZRb9k...| Customer Account|
|2WIRuoubMt80KRTze...| Customer Account|
|Y+xeMMwM4Lh/3tAiI...| Customer Account|
|2eGlPJhootHD4+Wqi...| Customer Account|
|x2IFZ35cZCni3H49P...| Customer Account|
+--------------------+-----------------+
only showing top

In [ ]:
# FIXED: Python version mismatch issue - using alternative approach
import os

# Fix Python version mismatch by setting environment variables
os.environ['PYSPARK_PYTHON'] = '/usr/bin/python3.10'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/usr/bin/python3.10'

print("\n🔧 Fixed Python version mismatch - set both driver and worker to Python 3.10")
print("🔍 Using alternative batch processing strategy to avoid RDD operations...")

# Alternative approach: Use DataFrame operations instead of RDD to avoid Python version issues
def process_senders_in_batches_fixed():
    """Process July senders using DataFrame operations instead of RDDs"""
    
    all_results = []
    
    # Use DataFrame row_number() instead of RDD zipWithIndex to avoid Python version issues
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number
    
    # Add row numbers using DataFrame operations
    window_spec = Window.orderBy("ac_from")
    df_july_senders_indexed = df_july_senders.withColumn("index", row_number().over(window_spec) - 1)
    df_july_senders_indexed.cache()
    
    print(f"✅ Created indexed DataFrame with {df_july_senders_indexed.count():,} records")
    
    for batch_num in range(total_batches):
        start_idx = batch_num * BATCH_SIZE
        end_idx = min((batch_num + 1) * BATCH_SIZE, july_senders_count)
        
        print(f"\n📦 Processing batch {batch_num + 1}/{total_batches} (accounts {start_idx} to {end_idx-1})")
        
        # Get batch using DataFrame operations
        batch_senders = df_july_senders_indexed.filter(
            (col("index") >= start_idx) & (col("index") < end_idx)
        ).select("ac_from")
        
        # Convert to Pandas to avoid Spark collect() issues with version mismatch
        try:
            batch_senders_pandas = batch_senders.toPandas()
            batch_senders_list = batch_senders_pandas['ac_from'].tolist()
        except Exception as e:
            print(f"⚠️  Pandas conversion failed, using alternative method: {str(e)[:50]}...")
            # Fallback: use take() instead of collect()
            batch_senders_list = [row.ac_from for row in batch_senders.take(BATCH_SIZE)]
        
        if not batch_senders_list:
            print(f"⚠️  Batch {batch_num + 1}: No data to process")
            continue
            
        print(f"📋 Batch {batch_num + 1}: Processing {len(batch_senders_list)} accounts")
        
        # Escape and create IN clause
        batch_senders_clean = [str(ac).replace("'", "''") for ac in batch_senders_list]
        batch_in_clause = "'" + "','".join(batch_senders_clean) + "'"
        
        # Create batch-specific query
        batch_query = f"""
        (SELECT a_c_reference, account_type_name 
            FROM {stixor_mbar_v_table} 
            WHERE a_c_reference IN ({batch_in_clause})) as batch_{batch_num}
        """
        
        # Optimized properties for batch processing
        batch_properties = {
            **properties,
            "fetchsize": "1000",        # Smaller to avoid memory issues
            "batchsize": "3000",        # Reduced batch size
            "queryTimeout": "180",      # Shorter timeout
            "socketTimeout": "180",     # Match query timeout
            "defaultRowFetchSize": "1000",
            "prepareThreshold": "1",
            "reWriteBatchedInserts": "true"
        }
        
        try:
            # Load batch data
            batch_result = spark.read.jdbc(
                url=jdbc_url,
                table=batch_query,
                properties=batch_properties
            )
            
            # Cache and count this batch
            batch_result.cache()
            batch_count = batch_result.count()
            print(f"✅ Batch {batch_num + 1}: Loaded {batch_count:,} account records")
            
            # Collect results for this batch
            all_results.append(batch_result)
            
            # Clean up every 10 batches to avoid memory accumulation
            if (batch_num + 1) % 10 == 0:
                print(f"🧹 Checkpoint: Processed {batch_num + 1} batches, {len(all_results)} successful")
            
        except Exception as e:
            print(f"❌ Error in batch {batch_num + 1}: {str(e)[:100]}...")
            continue
    
    return all_results

# Execute the fixed batch processing
print("\n Starting fixed batch processing (no RDD operations)...")
batch_results = process_senders_in_batches_fixed()

25/10/15 18:55:41 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
25/10/15 18:55:41 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:295)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:955)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:166)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:273)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:174)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:116)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.proce

# Fraud Table

In [11]:
fraud_table_name = "public.fraud"

df_fraud = spark.read.jdbc(
    url=jdbc_url,
    table=fraud_table_name,
    properties=properties
)

print("✅ Loaded public.fraud table")
df_fraud.show(5)

✅ Loaded public.fraud table


+-------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-----------+-------------+-------------------+-------------------+--------------------+
|complaint_num|   trans_id|    complaint_msisdn|        fraud_msisdn|       victim_msisdn|             ac_from|               ac_to|trx_amount|trx_channel|     trx_type|   created_datetime|  resolved_datetime|transaction_datetime|
+-------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-----------+-------------+-------------------+-------------------+--------------------+
|   COM2176544|74871448252|fvdT0uX2/YoR6/D2K...|wmQJL8FEmQvt6rPU8...|BASO2pY/4oQAtzaqQ...|BASO2pY/4oQAtzaqQ...|fvdT0uX2/YoR6/D2K...|      3900|        API|Transfer(C2C)|2025-02-11 13:06:06|2025-02-11 13:30:00| 2025-02-08 22:53:55|
|   COM2284237|76205288132|8xhAAusuCbt8bgI7z...|wmQJL8FEmQvt6rPU8...|8xhAAus

In [ ]:
from pyspark.sql.functions import avg, unix_timestamp

# Calculate average resolution time using transaction_datetime
df_fraud_with_resolution = df_fraud.filter(df_fraud.resolved_datetime.isNotNull())
avg_resolution_seconds = df_fraud_with_resolution.select(
    avg(unix_timestamp("resolved_datetime") - unix_timestamp("transaction_datetime")).alias("avg_resolution_seconds")
).collect()[0]["avg_resolution_seconds"]

avg_resolution_hours = avg_resolution_seconds / 3600 if avg_resolution_seconds is not None else None
print(f"Average resolution time: {avg_resolution_hours:.2f} hours" if avg_resolution_hours is not None else "No resolved cases found.")

Average resolution time: 103.13 hours


In [13]:
# Get distinct values for each column in df_fraud
distinct_complaint = df_fraud.select("complaint_msisdn").distinct()
distinct_fraud = df_fraud.select("fraud_msisdn").distinct()
distinct_victim = df_fraud.select("victim_msisdn").distinct()
distinct_ac_from = df_fraud.select("ac_from").distinct()
distinct_ac_to = df_fraud.select("ac_to").distinct()

# Save each to parquet
distinct_complaint.write.mode("overwrite").parquet("../data/distinct_complaint_msisdn")
distinct_fraud.write.mode("overwrite").parquet("../data/distinct_fraud_msisdn")
distinct_victim.write.mode("overwrite").parquet("../data/distinct_victim_msisdn")
distinct_ac_from.write.mode("overwrite").parquet("../data/distinct_ac_from")
distinct_ac_to.write.mode("overwrite").parquet("../data/distinct_ac_to")

# Print unique and total counts
print(f"complaint_msisdn: unique={distinct_complaint.count():,}, total={df_fraud.select('complaint_msisdn').count():,}")
print(f"fraud_msisdn: unique={distinct_fraud.count():,}, total={df_fraud.select('fraud_msisdn').count():,}")
print(f"victim_msisdn: unique={distinct_victim.count():,}, total={df_fraud.select('victim_msisdn').count():,}")
print(f"ac_from: unique={distinct_ac_from.count():,}, total={df_fraud.select('ac_from').count():,}")
print(f"ac_to: unique={distinct_ac_to.count():,}, total={df_fraud.select('ac_to').count():,}")

complaint_msisdn: unique=20,604, total=40,062
fraud_msisdn: unique=7,204, total=40,062
victim_msisdn: unique=20,598, total=40,062
ac_from: unique=20,581, total=40,062
ac_to: unique=14,862, total=40,062


# Account Types

## ac_from

In [27]:
stixor_mbar_v_table = "public.stixor_mbar_v"
ac_from_list = [row.ac_from for row in distinct_ac_from.collect()]
print(f"📊 Found {len(ac_from_list)} distinct ac_from values to filter")

ac_from_list_clean = [str(ac).replace("'", "''") for ac in ac_from_list]  # Escape single quotes
ac_from_in_clause = "'" + "','".join(ac_from_list) + "'"

sql_query = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({ac_from_in_clause})) as filtered_mbar
"""

# Create optimized properties for the filtered query
filtered_properties = {
    **properties,  # Inherit base properties
    "fetchsize": "5000",           # Smaller fetch size for filtered data
    "batchsize": "10000",          # Optimized batch size
    "queryTimeout": "600",         # Shorter timeout for filtered query
    "socketTimeout": "600",        # Match query timeout
    "defaultRowFetchSize": "5000", # Optimized fetch size
    "prepareThreshold": "1",       # Prepare statement immediately
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered data with SQL pushdown...")

df_mbar_filtered = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query,
    properties=filtered_properties
)

# Save results for future use
print("\n💾 Saving ac_from results...")
df_mbar_filtered.coalesce(5).write.mode("overwrite").parquet("../data/ac_from_accounts_with_types")
print("✅ Results saved to ../data/ac_from_accounts_with_types")


# Group by account_type_name and count the number of records for each type
df_mbar_filtered.groupBy("account_type_name").count().orderBy("count", ascending=False).show()


📊 Found 20581 distinct ac_from values to filter
🚀 Loading filtered data with SQL pushdown...

💾 Saving ac_from results...


25/10/15 17:34:55 WARN DAGScheduler: Broadcasting large task binary with size 1326.7 KiB


✅ Results saved to ../data/ac_from_accounts_with_types


25/10/15 17:34:55 WARN DAGScheduler: Broadcasting large task binary with size 1660.0 KiB


+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account|20223|
|Organization Account|  147|
|Utility Bills Acc...|   34|
|Payment Gateway A...|   31|
|                NULL|   10|
| Account for Alfalah|    1|
+--------------------+-----+



25/10/15 17:34:56 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


## ac_to

In [26]:
stixor_mbar_v_table = "public.stixor_mbar_v"
ac_to_list = [row.ac_to for row in distinct_ac_to.collect()]
print(f"📊 Found {len(ac_to_list)} distinct ac_to values to filter")

ac_to_list_clean = [str(ac).replace("'", "''") for ac in ac_to_list]  # Escape single quotes
ac_to_in_clause = "'" + "','".join(ac_to_list_clean) + "'"

sql_query_to = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({ac_to_in_clause})) as filtered_mbar_to
"""

filtered_properties_to = {
    **properties,
    "fetchsize": "5000",
    "batchsize": "10000",
    "queryTimeout": "600",
    "socketTimeout": "600",
    "defaultRowFetchSize": "5000",
    "prepareThreshold": "1",
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered ac_to data with SQL pushdown...")

df_mbar_filtered_to = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_to,
    properties=filtered_properties_to
)

# Save results for future use
print("\n💾 Saving ac_to results...")
df_mbar_filtered_to.coalesce(5).write.mode("overwrite").parquet("../data/ac_to_accounts_with_types")
print("✅ Results saved to ../data/ac_to_accounts_with_types")

df_mbar_filtered_to.groupBy("account_type_name").count().orderBy("count", ascending=False).show()


📊 Found 14862 distinct ac_to values to filter
🚀 Loading filtered ac_to data with SQL pushdown...

💾 Saving ac_to results...


25/10/15 17:34:10 WARN DAGScheduler: Broadcasting large task binary with size 1025.2 KiB


✅ Results saved to ../data/ac_to_accounts_with_types


25/10/15 17:34:10 WARN DAGScheduler: Broadcasting large task binary with size 1207.6 KiB


+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account|13887|
|Payment Gateway A...|  325|
|Utility Bills Acc...|  310|
|Organization Account|  268|
|                NULL|    5|
|SP Account for Is...|    1|
|Raast Settlement ...|    1|
|SP Account for Ac...|    1|
|1-Link Organizati...|    1|
+--------------------+-----+



25/10/15 17:34:11 WARN DAGScheduler: Broadcasting large task binary with size 1605.3 KiB


## fraud_msisdn

In [25]:
fraud_msisdn_list = [row.fraud_msisdn for row in distinct_fraud.collect()]
print(f"📊 Found {len(fraud_msisdn_list)} distinct fraud_msisdn values to filter")

fraud_msisdn_list_clean = [str(ac).replace("'", "''") for ac in fraud_msisdn_list]
fraud_msisdn_in_clause = "'" + "','".join(fraud_msisdn_list_clean) + "'"

sql_query_fraud = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({fraud_msisdn_in_clause})) as filtered_mbar_fraud
"""

filtered_properties_fraud = {
    **properties,
    "fetchsize": "5000",
    "batchsize": "10000",
    "queryTimeout": "600",
    "socketTimeout": "600",
    "defaultRowFetchSize": "5000",
    "prepareThreshold": "1",
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered fraud_msisdn data with SQL pushdown...")

df_mbar_filtered_fraud = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_fraud,
    properties=filtered_properties_fraud
)
# Save results for future use
print("\n💾 Saving fraud_msisdn results...")
df_mbar_filtered_fraud.coalesce(5).write.mode("overwrite").parquet("../data/fraud_accounts_with_types")
print("✅ Results saved to ../data/fraud_accounts_with_types")

df_mbar_filtered_fraud.groupBy("account_type_name").count().orderBy("count", ascending=False).show()



📊 Found 7204 distinct fraud_msisdn values to filter
🚀 Loading filtered fraud_msisdn data with SQL pushdown...

💾 Saving fraud_msisdn results...
✅ Results saved to ../data/fraud_accounts_with_types
✅ Results saved to ../data/fraud_accounts_with_types
+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account| 5276|
|Organization Account|   70|
|Payment Gateway A...|   63|
|                NULL|   57|
|Utility Bills Acc...|   51|
|Raast Settlement ...|    1|
|1-Link Organizati...|    1|
+--------------------+-----+

+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account| 5276|
|Organization Account|   70|
|Payment Gateway A...|   63|
|                NULL|   57|
|Utility Bills Acc...|   51|
|Raast Settlement ...|    1|
|1-Link Organizati...|    1|
+--------------------+-----+



## victim_msisdn

In [23]:
# Analysis for victim_msisdn - Account types of fraud victims
victim_msisdn_list = [row.victim_msisdn for row in distinct_victim.collect()]
print(f"📊 Found {len(victim_msisdn_list)} distinct victim_msisdn values to filter")

# Handle potential SQL injection and optimize for large lists
victim_msisdn_list_clean = [str(msisdn).replace("'", "''") for msisdn in victim_msisdn_list]
victim_msisdn_in_clause = "'" + "','".join(victim_msisdn_list_clean) + "'"

sql_query_victim = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({victim_msisdn_in_clause})) as filtered_mbar_victim
"""

# Create optimized properties for the filtered query
filtered_properties_victim = {
    **properties,  # Inherit base properties
    "fetchsize": "5000",           # Smaller fetch size for filtered data
    "batchsize": "10000",          # Optimized batch size
    "queryTimeout": "600",         # Shorter timeout for filtered query
    "socketTimeout": "600",        # Match query timeout
    "defaultRowFetchSize": "5000", # Optimized fetch size
    "prepareThreshold": "1",       # Prepare statement immediately
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered victim_msisdn data with SQL pushdown...")
print(f"📋 Query preview: {sql_query_victim[:100]}...")

df_mbar_filtered_victim = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_victim,
    properties=filtered_properties_victim
)

print(f"✅ Loaded {df_mbar_filtered_victim.count():,} filtered records for victim_msisdn")

# Group by account_type_name and count the number of records for each type
print("\n📊 Account Type Distribution for victim_msisdn (Fraud Victims):")
df_mbar_filtered_victim.groupBy("account_type_name").count().orderBy("count", ascending=False).show()

# Save results for future use
print("\n💾 Saving victim_msisdn results...")
df_mbar_filtered_victim.coalesce(5).write.mode("overwrite").parquet("../data/victim_accounts_with_types")
print("✅ Results saved to ../data/victim_accounts_with_types")

📊 Found 20598 distinct victim_msisdn values to filter
🚀 Loading filtered victim_msisdn data with SQL pushdown...
📋 Query preview: 
(SELECT a_c_reference, account_type_name 
    FROM public.stixor_mbar_v 
    WHERE a_c_reference IN...


25/10/15 17:32:42 WARN DAGScheduler: Broadcasting large task binary with size 1103.0 KiB
25/10/15 17:32:43 WARN DAGScheduler: Broadcasting large task binary with size 1661.4 KiB
25/10/15 17:32:43 WARN DAGScheduler: Broadcasting large task binary with size 1661.4 KiB


✅ Loaded 20,365 filtered records for victim_msisdn

📊 Account Type Distribution for victim_msisdn (Fraud Victims):


25/10/15 17:32:44 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/10/15 17:32:44 WARN DAGScheduler: Broadcasting large task binary with size 1327.6 KiB


+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account|20236|
|Organization Account|   52|
|Utility Bills Acc...|   32|
|Payment Gateway A...|   28|
|                NULL|   17|
+--------------------+-----+


💾 Saving victim_msisdn results...
✅ Results saved to ../data/victim_accounts_with_types
✅ Results saved to ../data/victim_accounts_with_types


## complaint_msisdn

In [24]:
# Analysis for complaint_msisdn - Account types of those who filed fraud complaints
complaint_msisdn_list = [row.complaint_msisdn for row in distinct_complaint.collect()]
print(f"📊 Found {len(complaint_msisdn_list)} distinct complaint_msisdn values to filter")

# Handle potential SQL injection and optimize for large lists
complaint_msisdn_list_clean = [str(msisdn).replace("'", "''") for msisdn in complaint_msisdn_list]
complaint_msisdn_in_clause = "'" + "','".join(complaint_msisdn_list_clean) + "'"

sql_query_complaint = f"""
(SELECT a_c_reference, account_type_name 
    FROM {stixor_mbar_v_table} 
    WHERE a_c_reference IN ({complaint_msisdn_in_clause})) as filtered_mbar_complaint
"""

# Create optimized properties for the filtered query
filtered_properties_complaint = {
    **properties,  # Inherit base properties
    "fetchsize": "5000",           # Smaller fetch size for filtered data
    "batchsize": "10000",          # Optimized batch size
    "queryTimeout": "600",         # Shorter timeout for filtered query
    "socketTimeout": "600",        # Match query timeout
    "defaultRowFetchSize": "5000", # Optimized fetch size
    "prepareThreshold": "1",       # Prepare statement immediately
    "reWriteBatchedInserts": "true"
}

print(f"🚀 Loading filtered complaint_msisdn data with SQL pushdown...")
print(f"📋 Query preview: {sql_query_complaint[:100]}...")

df_mbar_filtered_complaint = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_complaint,
    properties=filtered_properties_complaint
)

print(f"✅ Loaded {df_mbar_filtered_complaint.count():,} filtered records for complaint_msisdn")

# Group by account_type_name and count the number of records for each type
print("\n📊 Account Type Distribution for complaint_msisdn (Complaint Filers):")
df_mbar_filtered_complaint.groupBy("account_type_name").count().orderBy("count", ascending=False).show()

# Save results for future use
print("\n💾 Saving complaint_msisdn results...")
df_mbar_filtered_complaint.coalesce(5).write.mode("overwrite").parquet("../data/complaint_accounts_with_types")
print("✅ Results saved to ../data/complaint_accounts_with_types")

📊 Found 20604 distinct complaint_msisdn values to filter
🚀 Loading filtered complaint_msisdn data with SQL pushdown...
📋 Query preview: 
(SELECT a_c_reference, account_type_name 
    FROM public.stixor_mbar_v 
    WHERE a_c_reference IN...


25/10/15 17:32:53 WARN DAGScheduler: Broadcasting large task binary with size 1103.3 KiB
25/10/15 17:32:53 WARN DAGScheduler: Broadcasting large task binary with size 1661.9 KiB
25/10/15 17:32:53 WARN DAGScheduler: Broadcasting large task binary with size 1661.9 KiB


✅ Loaded 20,335 filtered records for complaint_msisdn

📊 Account Type Distribution for complaint_msisdn (Complaint Filers):


25/10/15 17:32:54 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/10/15 17:32:54 WARN DAGScheduler: Broadcasting large task binary with size 1328.0 KiB


+--------------------+-----+
|   account_type_name|count|
+--------------------+-----+
|    Customer Account|20195|
|Organization Account|   52|
|Utility Bills Acc...|   34|
|Payment Gateway A...|   30|
|                NULL|   24|
+--------------------+-----+


💾 Saving complaint_msisdn results...
✅ Results saved to ../data/complaint_accounts_with_types
✅ Results saved to ../data/complaint_accounts_with_types
